# Chapter 13 &mdash; The Church&ndash;Turing Thesis

**Concept 3 of the Chapter 13 decomposition:** *The Church–Turing Thesis*

All the equivalent notions of universal computability together define the limit of effective calculability.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter13-TM/Concept-Church-Turing-Thesis/Concept-Church-Turing-Thesis.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Many independent formalisms were proposed for "effectively calculable": Turing
machines, Church's $\lambda$-calculus, G&ouml;del's recursive functions, Post systems,
register machines, cellular automata, your laptop.

**All turn out to be equivalent**, and no one has ever proposed anything strictly more
powerful that is also mechanically realisable.

> **The Church&ndash;Turing thesis.** Everything effectively calculable is computable by a
> Turing machine.

It is a **thesis**, not a theorem: "effectively calculable" is an informal notion, so
there is nothing to prove against. What makes it convincing is the **convergence** of
so many different definitions on the same class.

Practical consequence: to show something is computable, you may describe an algorithm
in English. To show something is *not*, you argue about Turing machines.

## 2. Definitions

### One function, three formalisms

In [ ]:
# (1) a Turing machine that appends a 1 (unary successor)
Succ = md2mc('''TM
I : 1 ; 1 , R -> I
I : . ; 1 , R -> F
''')

# (2) the lambda-calculus style: Church numerals
def church(n):
    return lambda f: lambda x: x if n == 0 else church(n-1)(f)(f(x))
def unchurch(c):
    return c(lambda k: k + 1)(0)
def csucc(c):
    return lambda f: lambda x: f(c(f)(x))

# (3) a recursive function
def rsucc(n):
    return n + 1

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch13&nbsp;2.&nbsp;The Halting Decider $H$, and the Grader's Dilemma](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter13-TM/Concept-The-Halting-Decider/Concept-The-Halting-Decider.ipynb) &nbsp;&middot;&nbsp; [**Chapter 13** index](https://github.com/ganeshutah/Jove/blob/master/Chapter13-TM/README.md) &nbsp;&middot;&nbsp; [Ch13&nbsp;4.&nbsp;Why Universal Formalisms Matter: They Are Vehicles for Proofs](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter13-TM/Concept-Why-Universal-Formalisms-Matter/Concept-Why-Universal-Formalisms-Matter.ipynb)&nbsp;&rarr;

---

## 3. Tests

The Turing machine computes unary successor.

In [ ]:
for n in range(4):
    out = tm_tape(Succ, '1' * n if n else '.', fuel=50)
    print("  %d -> %r" % (n, out[0]))
    assert out[0].count('1') == n + 1

So do the $\lambda$-calculus and the recursive-function versions.

In [ ]:
for n in range(5):
    a = unchurch(csucc(church(n)))
    b = rsucc(n)
    c_ = len(tm_tape(Succ, '1' * n if n else '.', fuel=60)[0])
    print("  n=%d : lambda %d, recursive %d, TM %d" % (n, a, b, c_))
    assert a == b == c_ == n + 1

The convergence is the evidence.

In [ ]:
FORMALISMS = ["Turing machines", "lambda-calculus", "general recursive functions",
              "Post systems", "register machines", "cellular automata",
              "your laptop (with unbounded storage)"]
for f in FORMALISMS: print("  *", f)
print("\nAll provably equivalent.  None stronger.  That is the whole argument.")

It is a **thesis**: one side of it is informal, so it cannot be proved.

In [ ]:
print("computable by a TM        : a precise mathematical notion")
print("effectively calculable    : an INFORMAL notion about what people/machines can do")
print()
print("A proof would need both sides formal.  What we have instead is that every")
print("formalisation anyone has proposed lands in the same place.")

And the practical licence it grants.

In [ ]:
print("to show X IS computable     : describe an algorithm in English")
print("to show X is NOT computable : argue about Turing machines")
print()
print("Concept 4 explains why the second half needs the formal model.")

## 4. Exercises


1. Name a proposed model of computation that is **weaker** than a TM. Why?
2. Does quantum computation refute the thesis? (Careful: what does it change?)
3. What would a counterexample to the thesis even look like?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter13-TM/Concept-Church-Turing-Thesis')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')